<a href="https://colab.research.google.com/github/c-lydia/operating_system/blob/main/electronics_and_mcu_for_control_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electronics and Microcontrollers for Robotics Control

**Target platform:** STM32F103 (Cortex-M3)  
**Instructor:** Chheng Lydia

---

## Why control system need electronics

Let's begin with something familiar: a control loop.

```text
 reference --> [ control algorithm ] --> [ PWM / driver ] --> [ actuator ]
     ^              running on MCU                              |
     |                                                         v
     +------ [ encoder / ADC / interface ] <-- [ sensor ] <-- [ plant ]
```

In simulation, a controller can look beautifully simple. You define a reference, calculate an error, apply a PID equation, and get an output. On a real robot, that output has to become an electrical signal, pass through a driver, move a motor, and eventually return as a measurement.

That path is where many surprises live.

- The controller expects a sample every fixed interval, but the firmware updates irregularly.
- The encoder signal is noisy, or the input pin is floating.
- The motor driver needs more current than a GPIO can provide.
- The MCU resets when the motor starts because the supply dips or electrical noise couples into the logic.
- The control algorithm is fine, but the motor has a dead zone and does not move at low duty cycles.

So when a robot behaves badly, don't immediately blame the control law. First ask: **Did the hardware receive the signal we think we sent, and did the software read the measurement we think we received?**

### Where should the work happen?

A useful division of labour is:

- **MCU:** fast, predictable tasks such as PWM generation, encoder counting, sensor sampling, and low-level speed or position control.
- **Computer or single-board computer:** heavier tasks such as ROS 2, perception, mapping, planning, logging, and networking.

The boundary might be UART, CAN, USB, Ethernet, or another interface. The important point is that a Linux computer can be excellent at planning without being the best place to generate every timing-critical motor signal. Hardware timers on the MCU can continue producing a stable waveform while the CPU is doing other work.

---

## A few electrical ideas you must not skip

You do not need to become a power-electronics specialist before writing firmware. You do need enough electrical knowledge to avoid damaging hardware or misdiagnosing a problem.

### Voltage, current, resistance, and power

- **Voltage (V)** is the electrical potential difference between two points.
- **Current (I)** is the rate of flow of charge.
- **Resistance (R)** describes how strongly a component opposes current.
- **Power (P)** is the rate at which energy is transferred or dissipated.

Ohm's law is:

$$
V = IR
$$

Electrical power can be written as:

$$
P = VI
$$

For example, if a resistor has 1 kΩ across it and 3.3 V across its terminals, the current is $I=V/R=3.3/1000=3.3\text{ mA}$. This is the sort of calculation you should be comfortable doing before wiring a circuit.

### Logic levels are not power outputs

The STM32F103 uses 3.3 V logic. A GPIO pin is a signal pin, not a miniature motor power supply. Its current limits are constrained by the pin, port, and device specifications; do not design around the absolute-maximum rating as if it were a normal operating target.

**Never drive a motor, relay, or other high-current inductive load directly from a GPIO.** Use a suitable driver stage and follow its wiring and protection requirements.

Not every STM32F103 pin has the same electrical tolerance. Some pins are 5 V tolerant under specified conditions; others are not. Check the datasheet for the exact pin and mode before connecting a 5 V signal. “It worked once” is not an electrical specification.

### Ground, power, and noise

When two circuits exchange logic signals, they generally need a shared ground reference unless the interface is intentionally isolated. Without a common reference, a signal described as “3.3 V” may not mean 3.3 V to the receiving circuit.

At the same time, do not treat common ground as permission to route motor current through delicate logic wiring. Keep motor-current paths short and appropriately sized, keep logic and power wiring organized, and follow the driver and board guidance for supply decoupling and grounding.

Motors are noisy electrical loads. Switching their current can cause voltage spikes, supply dips, and electromagnetic interference. Use the motor driver's recommended inductive-load protection, decoupling, and power arrangement. Before blaming firmware, measure the supply rails and inspect the wiring.

### Inputs need defined states

A floating input can pick up noise and read unpredictably. Pull-up and pull-down resistors establish a known default state. The MCU may offer internal pulls, but whether they are appropriate depends on the signal, speed, noise environment, and circuit.

**Lab habit:** before connecting a signal, identify its voltage range, ground reference, direction, and current requirements.

---

## Processor, microprocessor, and microcontroller

People often use these terms loosely. For engineering work, it helps to distinguish them.

- **CPU / processor core:** the logic that fetches and executes instructions.
- **Microprocessor (MPU):** typically a high-performance processor that relies on external memory and other supporting components, often running a full operating system.
- **Microcontroller (MCU):** a chip that combines a CPU core with on-chip memory and peripherals such as timers, GPIO, ADC, and serial interfaces.

The STM32F103 is an MCU built around an ARM Cortex-M3 core. The core is only one part of the chip. The integrated timers, GPIO, communication peripherals, memory, and interrupt controller are what let a relatively small device interact with a robot.

A microprocessor running Linux may be much more capable at high-level computation, but a general-purpose OS does not automatically guarantee that a particular application instruction will execute at an exact instant. An MCU can use hardware timers and interrupts to make timing more predictable. That is why robots often use both kinds of processor.

---

## What is inside an STM32F103?

At a high level, think of the MCU as a CPU connected to memory and peripherals through buses.

```text
             +----------------------+
             |     Cortex-M3 CPU    |
             | registers / ALU      |
             +----------+-----------+
                        |
                 address/data buses
                        |
          +-------------+-------------+
          |             |             |
       [Flash]        [SRAM]      [Peripherals]
       program       variables    GPIO, timers,
       and constants stack/heap   ADC, USART, SPI,
                                  I2C, CAN, DMA...
                        |
                [Interrupt system]
```

### Flash, SRAM, and registers

- **Flash** is non-volatile memory. It stores the program and constants and retains them when power is removed.
- **SRAM** is volatile working memory for variables, buffers, stack, and heap.
- **CPU registers** are small, very fast storage locations inside the processor core.

Exact Flash and SRAM sizes depend on the STM32F103 part number. The F103 family contains multiple variants, so do not assume every chip has the same memory capacity. Check the marking on the chip and the matching datasheet.

### Memory-mapped peripherals

The MCU controls peripherals by reading and writing designated memory addresses. This is called **memory-mapped I/O**.

For example, firmware can write to a GPIO output register to change a pin's state, or write to a timer's compare register to change PWM duty cycle. The address is not ordinary RAM: it refers to a hardware register.

This is why peripheral access must follow the reference manual. A register name, bit position, or clock-enable path from a different STM32 family may look familiar but still be wrong for the F103.

### Polling and interrupts

With **polling**, software repeatedly checks whether something has happened. With an **interrupt**, hardware signals an event and the processor runs an interrupt service routine (ISR).

Interrupts are useful for time-sensitive events, but they are not magic. Keep ISRs short, avoid blocking delays inside them, and be deliberate about how data is shared between interrupt code and the main loop. `volatile` can be necessary for data shared with an ISR, but it does not by itself make multi-step operations atomic or solve every concurrency problem.

---

## Peripherals: the parts that make the MCU useful

You will meet these repeatedly in robotics:

| Peripheral | What it does | Example use |
|---|---|---|
| GPIO | Reads or drives digital signals | Enable pins, buttons, LEDs |
| Timers | Count clock ticks and generate/capture events | PWM, encoder counting, periodic control tick |
| ADC | Converts an analog voltage to a number | Current or analog sensor measurement |
| USART / UART | Serial communication | Debug output, MCU-to-computer link |
| SPI / I²C | Peripheral communication buses | IMUs, sensors, peripheral ICs |
| CAN | Robust multi-node communication bus | Motor controllers and robot subsystems |
| DMA | Transfers data between memory and peripherals with less CPU involvement | Repeated sampling and data movement |
| Watchdog | Resets the MCU if software fails to service it | Recovery from certain firmware hangs |

A peripheral can often perform its job while the CPU does something else. For example, a timer can generate PWM in hardware without the firmware manually toggling a pin for every edge. That is both more efficient and more consistent.

---

## Why use STM32 for control work?

Arduino, ESP32, and STM32 boards can all be useful. The point is not that one platform is universally best; it is that different projects have different priorities.

- **Arduino-style platforms** are approachable for simple experiments and quick prototypes.
- **ESP32** is attractive when integrated wireless networking is important.
- **STM32** provides a broad range of timers, ADCs, communication peripherals, and development options useful for embedded control.

The STM32F103 is an older but widely encountered Cortex-M3 family. It is capable of teaching GPIO, timers, interrupts, serial communication, and embedded C fundamentals. More advanced STM32 families offer additional motor-control features, so do not assume every feature available on a newer STM32 is present on the F103.

The trade-off is that STM32 development exposes more configuration choices. That can feel like extra work at first, but learning to understand the clock tree, pin mapping, timer configuration, and documentation is part of becoming an embedded engineer.

---

## From source code to running firmware

Firmware passes through several stages before it runs on the board.

```text
 C source files
      |
      v
  [Compiler]  ---> object files
      |                |
      +----------------+
               |
           [Linker]  ---> firmware.elf
               |
        [Programmer / debugger]
               |
          STM32F103 board
```

- The **compiler** translates C code into machine instructions.
- The **startup code** establishes the initial runtime environment.
- The **linker script** describes where code and data belong in memory.
- The **build system** coordinates compilation and linking.
- The **programmer/debugger** downloads firmware and helps inspect execution.

You can work with register-level CMSIS code, STM32 LL, or HAL. Each layer trades direct control for convenience:

- **Registers / CMSIS:** close to the hardware, useful for learning exactly what a peripheral does.
- **LL:** thin helper functions around peripheral operations.
- **HAL:** higher-level functions that simplify setup and improve readability, at the cost of hiding some details.

For this topic, the important thing is not to declare one layer morally superior. Use the layer your project supports, then inspect what it configures. If you use CubeMX-generated code, keep application code in the designated user-code sections so regeneration does not overwrite it.

---

## Read the documentation instead of guessing

Three documents are especially useful:

1. **Datasheet:** electrical limits, pinout, memory capacity, package details, and alternate-function mapping.
2. **Reference manual:** peripheral architecture, register definitions, bit meanings, and configuration sequences.
3. **Programming manual / Cortex-M documentation:** core behavior, instructions, exceptions, and interrupt handling.

Also use the board's own schematic or user guide. The chip datasheet tells you what the MCU can do; the board documentation tells you how that chip is wired on your particular board.

### A practical documentation workflow

Suppose you want PWM on a pin:

1. Find the pin on your board schematic or pinout.
2. Check the STM32F103 datasheet to confirm the pin's alternate functions and electrical constraints.
3. Open the timer chapter in the F103 reference manual.
4. Read the timer block diagram and PWM-mode description before diving into bit fields.
5. Configure the peripheral in your chosen toolchain.
6. Compare the generated or library code with the manual.
7. Measure the output and confirm it matches the calculation.

When using online tutorials, check that they target the STM32F1 family or a compatible device and library version. An STM32L4 or newer STM32 example may use different registers, clock-enable paths, GPIO configuration, or alternate-function setup. Tutorials are useful; the exact device documentation remains the authority.

---

## Timers and PWM: turning numbers into a signal

Now we can make the MCU produce a signal that we can measure.

**Pulse-width modulation (PWM)** switches a digital output between LOW and HIGH. The duty cycle is the fraction of each period that the output is HIGH. For many driver circuits, changing duty cycle changes the average power delivered to a load, although the exact motor response depends on the driver and motor.

```text
One PWM period:
HIGH  ┌───────┐       ┌───────┐
      │       │       │       │
LOW ──┘       └───────┘       └──
      <------ period -------->
      < HIGH >
      duty cycle = HIGH time / period
```

Two quantities matter:

- **Frequency** tells us how many periods occur per second.
- **Duty cycle** tells us what fraction of each period is HIGH.

For example, at 1 kHz, one period lasts 1 ms. At 20 kHz, one period lasts 50 μs. A 25% duty cycle at 20 kHz means the output is HIGH for 12.5 μs and LOW for 37.5 μs in each period.

### How a hardware timer generates PWM

A timer counts clock ticks. The auto-reload register (ARR) sets the period, while a capture/compare register (CCR) determines the point at which the output changes state. The prescaler (PSC) divides the timer's input clock.

For the common up-counting configuration:

$$
f_{\text{PWM}} = \frac{f_t}{(PSC + 1)(ARR + 1)}, t = timer
$$

The `+1` terms matter because the register values are zero-based. A prescaler value of 0 means divide by 1, not divide by 0.

For PWM mode 1, the duty cycle is approximately:

$$
D = \frac{CCR}{ARR+1}
$$

or as a percentage:

$$
D_{\%} = 100 \times \frac{CCR}{ARR+1}
$$

Exact edge behavior at 0% and 100% should be checked against the timer mode and configuration, but these formulas are the right starting point for ordinary edge-aligned PWM.

### Worked example — calculate before configuring

Suppose the timer clock has been confirmed as 72 MHz and we want 20 kHz PWM.

$$
ARR+1 = \frac{72,000,000}{20,000}=3600
$$

Choose `PSC = 0`, giving `ARR = 3599`.

For a 35% duty cycle:

$$
CCR \approx 0.35 \times 3600 = 1260
$$

So the starting configuration is `PSC = 0`, `ARR = 3599`, and `CCR = 1260`.

Do not copy these numbers blindly. First confirm the timer clock in your actual clock configuration. The STM32F103 clock tree has APB prescaler rules that can make a timer's clock differ from the bus clock. The formula is general; the input clock is a fact you must establish for your setup.

### Frequency and resolution

With a fixed timer clock, a higher PWM frequency means fewer timer counts per period, so there are fewer possible duty-cycle steps. A lower frequency gives more steps but may be unsuitable for the application. Motor-driver requirements, audible noise, switching losses, current ripple, and the motor's electrical characteristics all matter when selecting a carrier frequency.

20 kHz is a common choice in some motor-control applications because it is around the upper edge of typical human hearing, but it is not a universal rule. Follow the driver and project requirements.

---

## Practical implementation: make PWM, then measure it

Start with a safe output such as an LED or a disconnected logic-level test point. Do not begin by connecting an unverified PWM output to a motor power stage.

The exact steps depend on the board, firmware framework, and pin assignment. On an STM32F103, GPIO configuration and timer alternate-function mapping differ from newer STM32 families. Check the datasheet and reference manual for the exact pin and timer channel you intend to use.

### The basic workflow

1. Identify the board and exact STM32F103 part.
2. Confirm the clock configuration and timer input clock.
3. Select a pin that supports the desired timer channel, accounting for any timer remapping.
4. Enable the required GPIO and timer peripheral clocks.
5. Configure the pin for the appropriate alternate-function output mode.
6. Configure the timer prescaler, auto-reload value, PWM mode, and compare value.
7. Enable the channel output and start the timer.
8. Measure the signal before connecting it to a driver or other load.

Whether you use HAL or registers, you should be able to explain what each configuration item does. A successful build only proves that the code compiled; it does not prove that the pin is configured correctly or that the waveform is what you intended.

### What to verify

- Does the pin switch between the expected LOW and HIGH voltage?
- Is the period correct? At 20 kHz, it should be 50 μs.
- Is the HIGH time correct? At 35%, it should be about 17.5 μs.
- Does changing the compare value change duty cycle without unexpectedly changing frequency?
- Does the debugger show the register values you expected?

Use an oscilloscope when you need to see voltage over time. A logic analyzer is useful for digital timing and protocol decoding, but it does not replace voltage measurements when signal levels or noise are in question.

---

## 11. Debug systematically

When the output is wrong, resist the temptation to change several things at once. Work from the physical layer upward.

1. **Power and ground:** are the board and peripherals powered correctly?
2. **Clock:** is the MCU running at the frequency you assumed?
3. **Pin mapping:** does this pin support the chosen timer channel, and is remapping configured correctly?
4. **Peripheral clocks:** are the relevant GPIO and timer clocks enabled?
5. **Timer configuration:** are PSC, ARR, PWM mode, compare value, channel enable, and counter enable correct?
6. **Measurement setup:** is the probe connected to the correct point with a suitable ground reference?
7. **Application logic:** is later code overwriting the compare value or stopping the timer?

Common clues:

- **No waveform:** check the pin mode, alternate-function mapping, peripheral clock, channel enable, and timer enable.
- **Frequency is wrong by a factor of two:** check the actual timer clock and APB prescaler configuration.
- **Frequency is slightly wrong:** check the `+1` terms and the assumed clock frequency.
- **The board resets when a motor starts:** investigate supply dips, grounding, wiring, and motor-driver noise before changing the control algorithm.
- **The signal looks right but the motor does not turn:** the driver may be disabled, the motor may be below its start threshold, or the driver input logic may not match your assumptions.

Debugging is an experiment. Change one variable, predict what should happen, measure the result, and write down what you learned.

---

## Note

Embedded systems are easier to maintain when the assumptions are written down.

Before integrating a subsystem, document a small note:

```text
Subsystem: PWM output
Target: STM32F103 (exact board/part to be recorded)
Pin and timer channel: verified from datasheet
Timer input clock: measured/confirmed from clock configuration
Frequency: chosen value, with calculation
Duty-cycle input: range and meaning
Output voltage levels: verified for the selected pin
Dependencies: clock setup, GPIO setup, timer setup
Verification: scope or logic-analyzer result
Known limitations: pin remapping, driver requirements, etc.
```

Use version control to isolate work in a feature branch, make focused commits, and review changes before merging. Record not only what worked but also unexpected behavior, measurements, and the next test. A useful lab notebook lets another engineer reproduce your result without guessing what you connected or configured.

### Safety habits

- Never power a motor directly from an MCU GPIO.
- Verify pin voltage tolerance before connecting an external signal.
- Check polarity and ground reference before applying power.
- Use the motor driver's specified protection and wiring.
- Start with a safe, current-limited setup when testing power electronics.
- Keep the robot's wheels clear of the ground or the mechanism secured when testing motor outputs.
- Make sure an emergency stop or power-disconnect method is available before testing a moving robot.

---

## Checkpoint exercises

These exercises are intended to test understanding, not memorization.

1. **Trace the control loop.** Explain how a controller's numeric output becomes a physical motor command and how feedback returns to the MCU.
2. **Choose the processor.** Give one reason to run low-level motor control on an MCU and one reason to run perception or planning on a computer.
3. **Find the right document.** Identify the exact STM32F103 part and find its Flash/SRAM capacity, pin voltage limits, and alternate-function table.
4. **Calculate PWM.** Given a confirmed timer clock of 72 MHz, choose PSC and ARR for 20 kHz PWM, then calculate the CCR value for 25% duty.
5. **Predict before measuring.** Calculate the period and HIGH time, then compare them with a scope capture.
6. **Debug a missing waveform.** List the checks you would perform, in order, before rewriting the application.
7. **Investigate motor dead zone.** Why might a motor fail to start at low duty even though the PWM waveform is correct? Consider static friction, load, driver voltage drop, supply voltage, and current.
8. **Explain the code.** For each timer setting, explain which physical property it changes and how you would verify that change.

For each practical task, submit the calculation, the relevant documentation reference, a measurement or screenshot, and a short explanation of any mismatch.

---

## What comes next?

This topic establishes the connection between firmware, timing, and hardware. The natural sequence for a robotics-control specialization is:

1. **GPIO and interrupt fundamentals:** reliable inputs, outputs, and event handling.
2. **Timers and encoders:** position counting, speed estimation, and periodic sampling.
3. **ADC and sensors:** sampling, resolution, noise, and calibration.
4. **Motor-driver electronics:** H-bridges, enable/direction interfaces, protection, and safe switching.
5. **Real-time control loop:** sample timing, jitter, execution time, and non-blocking design.
6. **PID on the MCU:** discretization, saturation, anti-windup, and testing against the real plant.
7. **Communication:** UART/CAN and a clean interface between low-level firmware and higher-level robot software.

